In [1]:
import sys
import os

# Agregar el directorio padre al sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime


import cashia_core.data_channel.promok_datapipe_func as pkdp
import cashia_core.common_tools.pm_mxgeographic as pm_mxgeo

import mlp.promok_models_config as pkconfig
from mlp.dataframe_tracker import *


%matplotlib inline

In [2]:
import sklearn as sk

print("numpy:",np.__version__)
print("pandas:",pd.__version__)
print("seborn:",sns.__version__)
print("sklearn:",sk.__version__)
#print(":",.__version__)

numpy: 1.26.0
pandas: 1.5.3
seborn: 0.13.2
sklearn: 1.2.2


# 1 Procesar los datos

## 1.1 Lectura de los datos

### 1.1.1 Lectura de los datos de configuración

In [3]:
print('Configuration file: ', pkconfig.configuration_file_name)
print('Configuration : ', pkconfig.conf_column)
all_features = pd.read_excel("./config/" + pkconfig.configuration_file_name, index_col=0)

Configuration file:  CashIA_ConfFile.xlsx
Configuration :  RNV_Agt_CS


c:\Users\juan_\.conda\envs\cashia_env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [4]:
features = all_features[all_features[pkconfig.conf_column] == 'Sí']
features

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
8,No,Sí,No,No,No,Sí,No,Sí,Ciclo,Predictor,1,1.0,50.0,Cuantitativo,Entero,0,Crédito,NaN,NaN
9,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Monto,Predictor,1,900.0,20000.0,Cuantitativo,Entero,1,Crédito,NaN,x
11,Sí,Sí,Sí,No,Sí,Sí,Sí,Sí,Edad Solicitud,Predictor,1,18.0,100.0,Cuantitativo,Entero,0,Datos personales,Edad al hacer la solicitud,NaN
15,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Genero,Predictor,0,NaN,NaN,Categórico,Texto,0,Datos personales,NaN,NaN
18,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Dependientes,Predictor,1,0.0,10.0,Cuantitativo,Entero,0,Datos personales,NaN,x
34,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Ingresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de ingresos,NaN,x
35,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Egresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de egresos,NaN,x
52,No,Sí,No,No,No,Sí,No,Sí,Score_Ant,Predictor,1,0.0,5.0,Cuantitativo,Real,0,Promocash,NaN,NaN
53,No,Sí,No,No,No,Sí,No,Sí,Monto_Ant,Predictor,1,0.0,20000.0,Cuantitativo,Entero,1,Promocash,NaN,NaN


In [5]:
features_to_use = set(features['Característica'].values)
features_to_use

{'Ciclo',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Frecuencia_Ant',
 'Genero',
 'Incobrable',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Monto_Ant',
 'Score Agt',
 'Score_Ant'}

In [6]:
agregated_feautures = set(features[features['Sección'] == 'Data Agregation']['Característica'].values)
agregated_feautures

{'Mes inicial'}

### 1.1.2 Lectura de los datos de entrenamiento

In [7]:
print('Source file: ', pkconfig.source_file_name)
#data = pd.read_excel(pkconfig.source_file_name)
data = pd.read_csv("./data/" + pkconfig.source_file_name, index_col=0)

data_tracker = DataFrameEvolutionTracker(data)
data_tracker.to_dataframe()

if pkconfig.conf_column == "NV_Agt" or pkconfig.conf_column == "RNV_Agt":
    print("Agent score")
    data = data[data['Score Agt'] != 0]
else:
    print("Using CC score")

data_tracker.register_step(f"First filter for {pkconfig.conf_column}", data)
data_tracker.to_dataframe()

data.head()

Source file:  Data Set Cashia Entrenamiento.csv


C:\Users\juan_\AppData\Local\Temp\ipykernel_34536\725869802.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("./data/" + pkconfig.source_file_name, index_col=0)


Using CC score


,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0.000000,13,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.00000
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,0.000000,23,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.00000
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.0,2005,0.05882,561.0,17.0,0.00000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.0,2005,0.05882,561.0,17.0,0.05882
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.0,2107,0.05882,568.0,17.0,0.00000


# Filtros especiales

## Borramos los datos de créditos no aprobados

In [8]:
# Quitamos esta línea en diciembre 2025 ya que en teoría sólo me pasan los créditos aprobados
# data = data[data['Aprobado'] == 1]
# data.shape

## De acuerdo a la columna de configuración selecionamos los datos

In [9]:
# Caso 1: Renovaciones
if pkconfig.conf_column == "NV_Agt":
    data = data[data['Categoria'] == "NV"]
    
elif pkconfig.conf_column == "RNV_Agt":
    data = data[(data['Categoria'] == "RNV")]
    
elif pkconfig.conf_column == "NV_CC":
    data = data[(data['Categoria'] == "NV")]
    
elif pkconfig.conf_column == "RNV_CC":
    data = data[(data['Categoria'] == "RNV")]

elif pkconfig.conf_column == "NV_Agt_CS":
    data = data[data['Categoria'] == "NV"]
    
elif pkconfig.conf_column == "RNV_Agt_CS":
    data = data[(data['Categoria'] == "RNV")]
    
elif pkconfig.conf_column == "NV_CC_CS":
    data = data[(data['Categoria'] == "NV")]
    
elif pkconfig.conf_column == "RNV_CC_CS":
    data = data[(data['Categoria'] == "RNV")]
    
else:
    print("Invalid option")
    exit()

data_tracker.register_step(f"Second filter for {pkconfig.conf_column}", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


## 1.2 Explorar los datos

In [10]:
data_features = set(data.columns)
data_features

{'CP',
 'CP Aval',
 'Categoria',
 'Ciclo',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Fecha Inicial',
 'Frecuencia_Ant',
 'Genero',
 'Incobrable',
 'Ingresos',
 'Monto',
 'Monto_Ant',
 'Plazo',
 'Score Agt',
 'Score CC',
 'Score_Ant',
 'Unidad',
 'fecha_finalizada',
 'id_agente',
 'id_cliente',
 'id_credito_ant',
 'id_fecha_finalizada_actual',
 'id_fecha_finalizada_ant',
 'id_fecha_impresion_actual',
 'id_fecha_impresion_ant',
 'pagado_cred_ant',
 'pagos_realizados_actual',
 'referencia',
 'score',
 'score_agente',
 'sem_inicio',
 'semanas',
 'tarifa_ant',
 'vida_actual',
 'vida_ant'}

In [11]:
features_to_drop = list(data_features - features_to_use)
# Removemos de la lista de elementos a borrar 'Fecha Inicial ya que la necesitaremos para los intervalos de aprendizaje'
features_to_drop.remove('Fecha Inicial')
features_to_drop

['pagos_realizados_actual',
 'pagado_cred_ant',
 'vida_ant',
 'id_fecha_finalizada_ant',
 'tarifa_ant',
 'Categoria',
 'sem_inicio',
 'id_credito_ant',
 'fecha_finalizada',
 'Plazo',
 'Unidad',
 'id_fecha_impresion_actual',
 'CP Aval',
 'vida_actual',
 'id_fecha_impresion_ant',
 'id_cliente',
 'Score CC',
 'CP',
 'score_agente',
 'id_fecha_finalizada_actual',
 'id_agente',
 'semanas',
 'referencia',
 'score']

In [12]:
features

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
8,No,Sí,No,No,No,Sí,No,Sí,Ciclo,Predictor,1,1.0,50.0,Cuantitativo,Entero,0,Crédito,NaN,NaN
9,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Monto,Predictor,1,900.0,20000.0,Cuantitativo,Entero,1,Crédito,NaN,x
11,Sí,Sí,Sí,No,Sí,Sí,Sí,Sí,Edad Solicitud,Predictor,1,18.0,100.0,Cuantitativo,Entero,0,Datos personales,Edad al hacer la solicitud,NaN
15,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Genero,Predictor,0,NaN,NaN,Categórico,Texto,0,Datos personales,NaN,NaN
18,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Dependientes,Predictor,1,0.0,10.0,Cuantitativo,Entero,0,Datos personales,NaN,x
34,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Ingresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de ingresos,NaN,x
35,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Egresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de egresos,NaN,x
52,No,Sí,No,No,No,Sí,No,Sí,Score_Ant,Predictor,1,0.0,5.0,Cuantitativo,Real,0,Promocash,NaN,NaN
53,No,Sí,No,No,No,Sí,No,Sí,Monto_Ant,Predictor,1,0.0,20000.0,Cuantitativo,Entero,1,Promocash,NaN,NaN


In [13]:
int_features_list = features[(features['Sub tipo'] == "Entero") & (features['Sección'] != "Data Agregation")]
int_features_list = int_features_list['Característica'].unique()
int_features_list = list(int_features_list)

In [14]:
int_features_list

['Ciclo',
 'Monto',
 'Edad Solicitud',
 'Dependientes',
 'Ingresos',
 'Egresos',
 'Monto_Ant',
 'Incobrable']

In [15]:
all_features

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
1,No,No,No,No,No,No,No,No,Id Solicitud,Predictor,1,0.0,999999.0,Cuantitativo entero,Entero,0,NaN,NaN,NaN
2,Sí- Off,No,Sí- Off,Sí- Off,Sí- Off,No,Sí- Off,Sí- Off,Plazo,Predictor,1,9.0,27.0,Categórico,Entero,0,Crédito,NaN,NaN
3,No,No,Sí- Off,Sí- Off,No,No,Sí- Off,Sí- Off,Categoria,Predictor,0,NaN,NaN,Categórico,Texto,0,Crédito,NaN,NaN
4,Sí,Sí,Sí,Sí,No,No,No,No,Unidad,Predictor,0,NaN,NaN,Categórico,Texto,0,Crédito,NaN,x
5,No,No,No,No,No,No,No,No,Zona,Predictor,0,NaN,NaN,Categórico,Texto,0,Crédito,"Mapearlo a ""unidad"" ya que son muchos valores",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,Sí,Sí,Sí,Sí,No,No,No,No,CP_Longitud,Predictor,1,-180.0,180.0,Cuantitativo,Real,0,Data Agregation,NaN,x
90,Sí- Off,No,Sí,Sí,Sí- Off,No,No,No,CP_Aval_Latitud,Predictor,1,-90.0,90.0,Cuantitativo,Real,0,Data Agregation,NaN,x
91,Sí- Off,No,Sí,Sí,Sí- Off,No,No,No,CP_Aval_Longitud,Predictor,1,-180.0,180.0,Cuantitativo,Real,0,Data Agregation,NaN,x


In [16]:
if ('CP' not in features_to_use) and (('CP_Latitud' in features_to_use)| ('CP_Longitud' in features_to_use)):
    int_features_list.append('CP')
    
if ('CP Aval' not in features_to_use) and (('CP_Aval_Latitud' in features_to_use) | ('CP_Aval_Longitud' in features_to_use)):
    int_features_list.append('CP Aval')                                        

if ('Score CC' not in features_to_use) and 'Odds' in features_to_use:
    int_features_list.append('Score CC')

In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 64759 entries, 66494 to 121839
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   referencia                  64759 non-null  object 
 1   id_cliente                  64759 non-null  int64  
 2   Fecha Inicial               64759 non-null  object 
 3   fecha_finalizada            47887 non-null  object 
 4   Monto                       64759 non-null  int64  
 5   Edad Solicitud              64759 non-null  int64  
 6   Genero                      64759 non-null  object 
 7   Dependientes                64759 non-null  int64  
 8   CP                          64759 non-null  int64  
 9   CP Aval                     63947 non-null  object 
 10  Ingresos                    64759 non-null  float64
 11  Egresos                     64759 non-null  float64
 12  Score CC                    64759 non-null  object 
 13  Plazo                     

In [18]:
data.isnull().sum()

referencia                        0
id_cliente                        0
Fecha Inicial                     0
fecha_finalizada              16872
Monto                             0
Edad Solicitud                    0
Genero                            0
Dependientes                      0
CP                                0
CP Aval                         812
Ingresos                          0
Egresos                           0
Score CC                          0
Plazo                             0
Categoria                         0
Unidad                            0
Ciclo                             0
id_credito_ant                  143
Monto_Ant                       143
tarifa_ant                      143
pagado_cred_ant                 143
id_fecha_impresion_ant          143
id_fecha_finalizada_ant         150
id_fecha_impresion_actual         0
id_fecha_finalizada_actual        0
pagos_realizados_actual           0
vida_ant                        150
Frecuencia_Ant              

## 1.3 Limpiar los datos

In [19]:
quantitative_features = set(features[(features['Tipo de dato'] == 'Cuantitativo') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
quantitative_features

{'Ciclo',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Frecuencia_Ant',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Monto_Ant',
 'Score Agt',
 'Score_Ant'}

In [20]:
categorical_features = set(features[(features['Tipo de dato'] == 'Categórico') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
categorical_features

{'Genero'}

### 1.3.1 Limpiar ingresos de conyuge para los que tienen NA poner "Desconocido"

In [21]:
if 'Ingreso Conyuge' in features_to_use: 
    pkdp.clean_ingreso_conyuge(data)
data_tracker.register_step(f"Cleaning datos conyuge", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


### 1.3.1b

In [22]:
#pk.clean_incobrable(data)

### 1.3.2 Poner los egresos igual a sus ingresos a aquellos que tienen vacío en egresos

In [23]:
pkdp.clean_egresos(data)
data_tracker.register_step(f"Cleaning egresos", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [24]:
data.isnull().sum()

referencia                        0
id_cliente                        0
Fecha Inicial                     0
fecha_finalizada              16872
Monto                             0
Edad Solicitud                    0
Genero                            0
Dependientes                      0
CP                                0
CP Aval                         812
Ingresos                          0
Egresos                           0
Score CC                          0
Plazo                             0
Categoria                         0
Unidad                            0
Ciclo                             0
id_credito_ant                  143
Monto_Ant                       143
tarifa_ant                      143
pagado_cred_ant                 143
id_fecha_impresion_ant          143
id_fecha_finalizada_ant         150
id_fecha_impresion_actual         0
id_fecha_finalizada_actual        0
pagos_realizados_actual           0
vida_ant                        150
Frecuencia_Ant              

In [25]:
raw_data = features_to_use-agregated_feautures
raw_data

{'Ciclo',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Frecuencia_Ant',
 'Genero',
 'Incobrable',
 'Ingresos',
 'Monto',
 'Monto_Ant',
 'Score Agt',
 'Score_Ant'}

In [26]:
data[list(raw_data)].isnull().sum()

Monto               0
Frecuencia_Ant      0
Egresos             0
Ciclo               0
Dependientes        0
Score Agt         878
Edad Solicitud      0
Genero              0
Ingresos            0
Monto_Ant         143
Incobrable          0
Score_Ant           0
dtype: int64

### 1.3.3. Llenamos con "Unknown" los valores de columnas categóricas vacias

In [27]:
#data[list(categorical_features-agregated_feautures)] = data[list(categorical_features-agregated_feautures)].fillna('Unknown')

### 1.3.4 Borrar renglones con datos nulos en las columnas que se usarán en el modelo

In [28]:
data = data.dropna(subset=list(features_to_use-agregated_feautures))
data_tracker.register_step(f"Cleaning for columns with null", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [29]:
data.isnull().sum()

referencia                        0
id_cliente                        0
Fecha Inicial                     0
fecha_finalizada              16119
Monto                             0
Edad Solicitud                    0
Genero                            0
Dependientes                      0
CP                                0
CP Aval                         782
Ingresos                          0
Egresos                           0
Score CC                          0
Plazo                             0
Categoria                         0
Unidad                            0
Ciclo                             0
id_credito_ant                    0
Monto_Ant                         0
tarifa_ant                        0
pagado_cred_ant                   0
id_fecha_impresion_ant            0
id_fecha_finalizada_ant           7
id_fecha_impresion_actual         0
id_fecha_finalizada_actual        0
pagos_realizados_actual           0
vida_ant                          7
Frecuencia_Ant              

### 1.3.5 Convertir a entero las columnas que deberían ser entero y no lo son

In [30]:
# 1) Convertir a numérico con NaN donde haya "NC" o vacío
data[int_features_list] = data[int_features_list].apply(
    pd.to_numeric, errors='coerce', downcast='integer'
)
data[int_features_list].isnull().sum()

Ciclo             0
Monto             0
Edad Solicitud    0
Dependientes      0
Ingresos          0
Egresos           0
Monto_Ant         0
Incobrable        0
dtype: int64

In [31]:
# 2) Eliminar filas con NaN en esas columnas
data = data.dropna(subset=int_features_list)

# 3) Forzar a int (si quieres asegurarte que sean int "normales")
data[int_features_list] = data[int_features_list].astype(int)

In [32]:
data_tracker.register_step(f"Cleaning int features with dropna", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


### 1.3.5 Convertir a float (Real) las columnas que deberían ser flot y no lo son

In [33]:
float_features_list = features[(features['Sub tipo'] == "Real") & (features['Sección'] != "Data Agregation")]['Característica'].unique()
float_features_list = list(float_features_list)

In [34]:
# if ('Score CC' not in features_to_use) and ('Odds' in features_to_use):
#     float_features_list.append('Score CC')    
# display(float_features_list)

In [35]:
# Convertir todo a floats (con NaN donde haya errores)
data[float_features_list] = data[float_features_list].apply(
    lambda col: pd.to_numeric(col, errors='coerce')
)

# Asegurar float64
data[float_features_list] = data[float_features_list].astype(float)

# Eliminar filas con problemas
data = data.dropna(subset=float_features_list)

print(data[float_features_list].info())
data_tracker.register_step(f"Cleaning na floats", data)
data_tracker.to_dataframe()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 63739 entries, 66494 to 121839
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Score_Ant       63739 non-null  float64
 1   Frecuencia_Ant  63739 non-null  float64
 2   Score Agt       63739 non-null  float64
dtypes: float64(3)
memory usage: 1.9 MB
None


,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
7,Cleaning na floats,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


### 1.3.6 Identificar características cuantitativas y cualitativas

In [36]:
for feature in quantitative_features : 
    if feature not in agregated_feautures:
        pkdp.categorical_print_values(feature, data)
        print("-----------------------")

Monto
5000     9367
7000     6153
10000    4677
6000     4550
8000     3891
         ... 
12300       1
2200        1
13600       1
12600       1
2800        1
Name: Monto, Length: 106, dtype: int64
-----------------------
Frecuencia_Ant
1.063784    1504
1.063783    1231
1.134703     752
1.060484     743
1.063516     707
            ... 
1.116147       1
1.433985       1
5.670412       1
1.424480       1
1.201922       1
Name: Frecuencia_Ant, Length: 14750, dtype: int64
-----------------------
Egresos
5000     6258
6000     5858
4000     4651
0        4087
8000     3591
         ... 
16136       1
15423       1
12863       1
5423        1
6458        1
Name: Egresos, Length: 665, dtype: int64
-----------------------
Ciclo
2     24938
3     14031
4      8424
5      5340
6      3506
7      2384
8      1649
9      1146
10      770
11      524
12      371
13      239
14      154
15      101
16       61
17       43
18       23
19       13
20       10
21        4
22        2
23        2
24  

In [37]:
data.head(5)

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.0,2005,0.05882,561.0,17.0,0.00000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.0,2005,0.05882,561.0,17.0,0.05882
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.0,2107,0.05882,568.0,17.0,0.00000
66533,k6653300198181,19818,30/12/2022,09/05/2023,7000,35,Masculino,0,62755,62755.0,...,1.202552,18,0,0.20454,0.0,2204,0.13529,561.0,17.0,0.00000
69435,k6943500275164,27516,23/01/2023,04/06/2023,6000,65,Femenino,0,94410,94410.0,...,1.482173,19,0,0.20454,0.0,2205,0.05882,564.0,17.0,0.00000


In [38]:
for feature in categorical_features :
    if feature not in agregated_feautures:
        pkdp.categorical_print_values(feature, data)
        print("-----------------------")

Genero
Femenino     47607
Masculino    16132
Name: Genero, dtype: int64
-----------------------


### 1.3.7 Limpiar datos numéricos

In [39]:
numeric_features = features[(features['Numérico'] == 1) & (features['Sección'] != 'Data Agregation')]  
display(numeric_features)

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
8,No,Sí,No,No,No,Sí,No,Sí,Ciclo,Predictor,1,1.0,50.0,Cuantitativo,Entero,0,Crédito,NaN,NaN
9,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Monto,Predictor,1,900.0,20000.0,Cuantitativo,Entero,1,Crédito,NaN,x
11,Sí,Sí,Sí,No,Sí,Sí,Sí,Sí,Edad Solicitud,Predictor,1,18.0,100.0,Cuantitativo,Entero,0,Datos personales,Edad al hacer la solicitud,NaN
18,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Dependientes,Predictor,1,0.0,10.0,Cuantitativo,Entero,0,Datos personales,NaN,x
34,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Ingresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de ingresos,NaN,x
35,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Egresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de egresos,NaN,x
52,No,Sí,No,No,No,Sí,No,Sí,Score_Ant,Predictor,1,0.0,5.0,Cuantitativo,Real,0,Promocash,NaN,NaN
53,No,Sí,No,No,No,Sí,No,Sí,Monto_Ant,Predictor,1,0.0,20000.0,Cuantitativo,Entero,1,Promocash,NaN,NaN
54,No,Sí,No,Sí,No,Sí,No,Sí,Frecuencia_Ant,Predictor,1,0.0,27.0,Cuantitativo,Real,0,Promocash,NaN,NaN


In [40]:
pkdp.print_sizes(data)
for index, row in numeric_features.iterrows():
    print(f"Column {row['Característica']} max value found: {data[row['Característica']].max()}")
    pkdp.clean_numeric_data(data, row['Característica'], row['Min'], row['Max'])
    print("=================================================")
pkdp.print_sizes(data)

Rows: 63739
Columns: 37
Max index: 264498
Column Ciclo max value found: 26
..=================================================
Column Monto max value found: 17000
..=================================================
Column Edad Solicitud max value found: 73
..=================================================
Column Dependientes max value found: 10
..=================================================
Column Ingresos max value found: 150000
..Total of incorrect registers: 102
For example: minvalue:0.0 maxvalue:50000.0 found 68000
Column Egresos max value found: 180000
..Total of incorrect registers: 100
For example: minvalue:0.0 maxvalue:50000.0 found 90000
Column Score_Ant max value found: 2.57528
..=================================================
Column Monto_Ant max value found: 17000
..=================================================
Column Frecuencia_Ant max value found: 27.0
..=================================================
Column Score Agt max value found: 4.46446
..Total of inc

In [41]:
data_tracker.register_step(f"Cleaning numeric data min max", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
7,Cleaning na floats,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
8,Cleaning numeric data min max,63535,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [42]:
data.head()

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.0,2005,0.05882,561.0,17.0,0.00000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.0,2005,0.05882,561.0,17.0,0.05882
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.0,2107,0.05882,568.0,17.0,0.00000
66533,k6653300198181,19818,30/12/2022,09/05/2023,7000,35,Masculino,0,62755,62755.0,...,1.202552,18,0,0.20454,0.0,2204,0.13529,561.0,17.0,0.00000
69435,k6943500275164,27516,23/01/2023,04/06/2023,6000,65,Femenino,0,94410,94410.0,...,1.482173,19,0,0.20454,0.0,2205,0.05882,564.0,17.0,0.00000


# 2 Data agregation

## 2.1 Agregar $Deficit = Ingresos - Egresos$

In [43]:
if 'Deficit' in quantitative_features:
    data['Deficit'] = data['Ingresos']-data['Egresos']
#quantitative_features.add('Deficit')

## 2.2 Agregar ingreso per capita

In [44]:
if 'IngresoPercapita' in  quantitative_features:
    data['IngresoPercapita'] = data['Ingresos']/(data['Dependientes']+1)
#quantitative_features.add('IngresoPercapita')

## 2.2 Agregar la zona a partir del Codigo Postal

In [45]:
#if 'TipoZona' in categorical_features:
#    CP_to_Zone = pd.read_excel("./CP_To_Zone.xlsx", index_col=0)
#    CP_Zone_set = set(CP_to_Zone.index)
#    data['TipoZona'] = data.apply(lambda x: CP_to_Zone.loc[x['CP']].Zona if x['CP'] in CP_Zone_set 
#                                  else "Unknown", axis=1)
#    data.head(5)

In [46]:
cp_manager = pm_mxgeo.MX_CP_manager('./mlp/inputs/MX_CP_CoordZone.csv')

In [47]:
data.columns

Index(['referencia', 'id_cliente', 'Fecha Inicial', 'fecha_finalizada',
       'Monto', 'Edad Solicitud', 'Genero', 'Dependientes', 'CP', 'CP Aval',
       'Ingresos', 'Egresos', 'Score CC', 'Plazo', 'Categoria', 'Unidad',
       'Ciclo', 'id_credito_ant', 'Monto_Ant', 'tarifa_ant', 'pagado_cred_ant',
       'id_fecha_impresion_ant', 'id_fecha_finalizada_ant',
       'id_fecha_impresion_actual', 'id_fecha_finalizada_actual',
       'pagos_realizados_actual', 'vida_ant', 'Frecuencia_Ant', 'vida_actual',
       'Incobrable', 'score', 'Score_Ant', 'id_agente', 'score_agente',
       'sem_inicio', 'semanas', 'Score Agt'],
      dtype='object')

In [48]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.00000,2005,0.05882,561.0,17.0,0.000000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.00000,2005,0.05882,561.0,17.0,0.058820
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.00000,2107,0.05882,568.0,17.0,0.000000
66533,k6653300198181,19818,30/12/2022,09/05/2023,7000,35,Masculino,0,62755,62755.0,...,1.202552,18,0,0.20454,0.00000,2204,0.13529,561.0,17.0,0.000000
69435,k6943500275164,27516,23/01/2023,04/06/2023,6000,65,Femenino,0,94410,94410.0,...,1.482173,19,0,0.20454,0.00000,2205,0.05882,564.0,17.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112748,k11274800032300,32300,27/11/2023,28/11/2024,5500,39,Femenino,0,72803,72848.0,...,1.063784,53,1,0.32537,0.20456,P1594A,0.61822,657.0,5.0,0.846870
113044,k11304429994,29994,28/11/2023,11/09/2024,7500,31,Femenino,3,72850,72850.0,...,1.385352,41,1,0.49302,0.20455,P1594A,0.27545,644.0,7.0,0.833420
118999,k118999000052009,52009,10/01/2024,NaN,4500,48,Femenino,0,74180,74180.0,...,1.035164,98,1,0.44247,0.23010,P1594A,0.24062,664.0,27.0,0.802422


In [49]:
if 'TipoZona' in categorical_features:
    data['index'] = data.index
    print(data.head(5))
    data = data.merge(cp_manager.cp_base[['CP', 'TipoZona']], left_on='CP', right_on='CP', how='left').fillna("Fake")
    data = data.set_index(['index'])

In [50]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.00000,2005,0.05882,561.0,17.0,0.000000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.00000,2005,0.05882,561.0,17.0,0.058820
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.00000,2107,0.05882,568.0,17.0,0.000000
66533,k6653300198181,19818,30/12/2022,09/05/2023,7000,35,Masculino,0,62755,62755.0,...,1.202552,18,0,0.20454,0.00000,2204,0.13529,561.0,17.0,0.000000
69435,k6943500275164,27516,23/01/2023,04/06/2023,6000,65,Femenino,0,94410,94410.0,...,1.482173,19,0,0.20454,0.00000,2205,0.05882,564.0,17.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112748,k11274800032300,32300,27/11/2023,28/11/2024,5500,39,Femenino,0,72803,72848.0,...,1.063784,53,1,0.32537,0.20456,P1594A,0.61822,657.0,5.0,0.846870
113044,k11304429994,29994,28/11/2023,11/09/2024,7500,31,Femenino,3,72850,72850.0,...,1.385352,41,1,0.49302,0.20455,P1594A,0.27545,644.0,7.0,0.833420
118999,k118999000052009,52009,10/01/2024,NaN,4500,48,Femenino,0,74180,74180.0,...,1.035164,98,1,0.44247,0.23010,P1594A,0.24062,664.0,27.0,0.802422


In [51]:
data.columns

Index(['referencia', 'id_cliente', 'Fecha Inicial', 'fecha_finalizada',
       'Monto', 'Edad Solicitud', 'Genero', 'Dependientes', 'CP', 'CP Aval',
       'Ingresos', 'Egresos', 'Score CC', 'Plazo', 'Categoria', 'Unidad',
       'Ciclo', 'id_credito_ant', 'Monto_Ant', 'tarifa_ant', 'pagado_cred_ant',
       'id_fecha_impresion_ant', 'id_fecha_finalizada_ant',
       'id_fecha_impresion_actual', 'id_fecha_finalizada_actual',
       'pagos_realizados_actual', 'vida_ant', 'Frecuencia_Ant', 'vida_actual',
       'Incobrable', 'score', 'Score_Ant', 'id_agente', 'score_agente',
       'sem_inicio', 'semanas', 'Score Agt'],
      dtype='object')

## 2.3 Agregar "Tiene aval" a partir de CP_Aval

In [52]:
if 'Tiene aval' in categorical_features:
    data['Tiene aval'] = data.apply(lambda x: 0 if pd.isnull(x['CP Aval']) else 1, axis=1)

## 2.3 Agregar Latitud y Longitud a partir del CP

In [53]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.00000,2005,0.05882,561.0,17.0,0.000000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.00000,2005,0.05882,561.0,17.0,0.058820
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.00000,2107,0.05882,568.0,17.0,0.000000
66533,k6653300198181,19818,30/12/2022,09/05/2023,7000,35,Masculino,0,62755,62755.0,...,1.202552,18,0,0.20454,0.00000,2204,0.13529,561.0,17.0,0.000000
69435,k6943500275164,27516,23/01/2023,04/06/2023,6000,65,Femenino,0,94410,94410.0,...,1.482173,19,0,0.20454,0.00000,2205,0.05882,564.0,17.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112748,k11274800032300,32300,27/11/2023,28/11/2024,5500,39,Femenino,0,72803,72848.0,...,1.063784,53,1,0.32537,0.20456,P1594A,0.61822,657.0,5.0,0.846870
113044,k11304429994,29994,28/11/2023,11/09/2024,7500,31,Femenino,3,72850,72850.0,...,1.385352,41,1,0.49302,0.20455,P1594A,0.27545,644.0,7.0,0.833420
118999,k118999000052009,52009,10/01/2024,NaN,4500,48,Femenino,0,74180,74180.0,...,1.035164,98,1,0.44247,0.23010,P1594A,0.24062,664.0,27.0,0.802422


In [54]:
if features_to_use.intersection(pkdp.latitud_longitud_features_set):
    
    if 'CP_Latitud'in features_to_use: 
        print("Seting CP_Latitud")
        data = cp_manager.set_geographic_coordinate(data,'CP', 'CP_Latitud', 'Latitud')
        print(data.columns)
        print("-----------------------")
    
    if 'CP_Longitud' in features_to_use: 
        print("Seting CP_Longitud")
        data = cp_manager.set_geographic_coordinate(data,'CP', 'CP_Longitud', 'Longitud')
        print(data.columns)
        print("-----------------------")

    if 'CP_Aval_Latitud' in features_to_use: 
        data = cp_manager.set_geographic_coordinate(data,'CP Aval','CP_Aval_Latitud', 'Latitud')
        print(data.columns)
        print("-----------------------")

    if 'CP_Aval_Longitud' in features_to_use: 
        data = cp_manager.set_geographic_coordinate(data,'CP Aval','CP_Aval_Longitud', 'Longitud')
        print(data.columns)
        print("-----------------------")

In [55]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.00000,2005,0.05882,561.0,17.0,0.000000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.00000,2005,0.05882,561.0,17.0,0.058820
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.00000,2107,0.05882,568.0,17.0,0.000000
66533,k6653300198181,19818,30/12/2022,09/05/2023,7000,35,Masculino,0,62755,62755.0,...,1.202552,18,0,0.20454,0.00000,2204,0.13529,561.0,17.0,0.000000
69435,k6943500275164,27516,23/01/2023,04/06/2023,6000,65,Femenino,0,94410,94410.0,...,1.482173,19,0,0.20454,0.00000,2205,0.05882,564.0,17.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112748,k11274800032300,32300,27/11/2023,28/11/2024,5500,39,Femenino,0,72803,72848.0,...,1.063784,53,1,0.32537,0.20456,P1594A,0.61822,657.0,5.0,0.846870
113044,k11304429994,29994,28/11/2023,11/09/2024,7500,31,Femenino,3,72850,72850.0,...,1.385352,41,1,0.49302,0.20455,P1594A,0.27545,644.0,7.0,0.833420
118999,k118999000052009,52009,10/01/2024,NaN,4500,48,Femenino,0,74180,74180.0,...,1.035164,98,1,0.44247,0.23010,P1594A,0.24062,664.0,27.0,0.802422


In [56]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 63535 entries, 66494 to 121839
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   referencia                  63535 non-null  object 
 1   id_cliente                  63535 non-null  int64  
 2   Fecha Inicial               63535 non-null  object 
 3   fecha_finalizada            47461 non-null  object 
 4   Monto                       63535 non-null  int32  
 5   Edad Solicitud              63535 non-null  int32  
 6   Genero                      63535 non-null  object 
 7   Dependientes                63535 non-null  int32  
 8   CP                          63535 non-null  int64  
 9   CP Aval                     62753 non-null  object 
 10  Ingresos                    63535 non-null  int32  
 11  Egresos                     63535 non-null  int32  
 12  Score CC                    63535 non-null  object 
 13  Plazo                     

## 2.4 Agregar los odds de circulo de crédito

In [57]:
if ('Odds' in quantitative_features):
    score_bins = [-1, 235, 406, 433, 451, 465, 477, 489, 499, 509, 518, 528, 538, 548, 559, 571, 584, 598, 614, 634, 664, 861]
    odds = [0, 0.90, 1.20, 1.30, 1.50, 1.60, 1.80, 1.90, 2.00, 2.10, 2.20, 2.60, 2.75, 2.90, 3.20, 3.50, 4.00, 4.60, 5.30, 6.10, 8.30]

    data['Odds'] = pd.cut(data['Score CC'], score_bins, labels=odds)
    data['Odds'] = data['Odds'].astype(float)
    print(data[['Score CC', 'Odds']])

In [58]:
quantitative_features

{'Ciclo',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Frecuencia_Ant',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Monto_Ant',
 'Score Agt',
 'Score_Ant'}

In [59]:
data.columns

Index(['referencia', 'id_cliente', 'Fecha Inicial', 'fecha_finalizada',
       'Monto', 'Edad Solicitud', 'Genero', 'Dependientes', 'CP', 'CP Aval',
       'Ingresos', 'Egresos', 'Score CC', 'Plazo', 'Categoria', 'Unidad',
       'Ciclo', 'id_credito_ant', 'Monto_Ant', 'tarifa_ant', 'pagado_cred_ant',
       'id_fecha_impresion_ant', 'id_fecha_finalizada_ant',
       'id_fecha_impresion_actual', 'id_fecha_finalizada_actual',
       'pagos_realizados_actual', 'vida_ant', 'Frecuencia_Ant', 'vida_actual',
       'Incobrable', 'score', 'Score_Ant', 'id_agente', 'score_agente',
       'sem_inicio', 'semanas', 'Score Agt'],
      dtype='object')

## 2.5 Agregar el número de mes en que se solicita el crédito

In [60]:
if ('Mes inicial' in quantitative_features):
    data['Fecha Inicial'] = pd.to_datetime(data['Fecha Inicial'], format='%d/%m/%Y')
    data['Mes inicial'] = data['Fecha Inicial'].dt.month

## 2.6 Transformación de columnas categóricas representadas como números

### 2.4.1 Mapear los estudios del cliente de un id entero a su significado

In [61]:
if 'Estudios' in features_to_use:
    nivel_educativo = {
        0: 'Sin Estudios',
        1: 'Primaria',
        2: 'Secundaria',
        3: 'Preparatoria',
        4: 'Técnica',
        5: 'Licenciatura',
        6: 'Maestría',
        7: 'Carrera Trunca'
    }
    data['Estudios'] = data['Estudios'].map(nivel_educativo)

In [62]:
data['Plazo'].head(5)

id_solicitud
66494    17 Sem
66857    17 Sem
72961    17 Sem
66533    17 Sem
69435    17 Sem
Name: Plazo, dtype: object

In [63]:
if 'Plazo' in features_to_use:
    plazo_pago = {
        9: '9 Sem', 
        13: '13 Sem',
        16: '16 Sem',  
        17: '17 Sem',
        26: '26 Sem', 
        27: '27 Sem', 
    }
    data['Plazo'] = data['Plazo'].map(plazo_pago)

In [64]:
data['Plazo'].head(5)

id_solicitud
66494    17 Sem
66857    17 Sem
72961    17 Sem
66533    17 Sem
69435    17 Sem
Name: Plazo, dtype: object

## 2.7 Usar solamente un numero N de dígitos en el código postal

In [65]:
#if pkconfig.zip_digits <= 4:
#    if 'CP' in features_to_use: pk.transform_CP(data,'CP', pkconfig.zip_digits)
#    if 'CP Aval' in features_to_use: pk.transform_CP(data,'CP Aval', pkconfig.zip_digits)

## Borrar las columnas que no serán usados

In [66]:
data = data.drop(labels=features_to_drop, axis=1)
data.head()

,Fecha Inicial,Monto,Edad Solicitud,Genero,Dependientes,Ingresos,Egresos,Ciclo,Monto_Ant,Frecuencia_Ant,Incobrable,Score_Ant,Score Agt,Mes inicial
id_solicitud,,,,,,,,,,,,,,
66494,2022-12-30,11000,24,Femenino,0,12400,1000,8,9000,1.063799,0,0.0,0.00000,12
66857,2023-01-03,14000,45,Femenino,0,7800,5300,8,12000,1.063799,0,0.0,0.05882,1
72961,2023-02-21,5000,29,Femenino,0,12000,4000,2,3000,1.198630,1,0.0,0.00000,2
66533,2022-12-30,7000,35,Masculino,0,7000,200,4,5000,1.202552,0,0.0,0.00000,12
69435,2023-01-23,6000,65,Femenino,0,10000,3500,3,5000,1.482173,0,0.0,0.00000,1


In [67]:
pkdp.print_sizes(data)

Rows: 63535
Columns: 14
Max index: 264498


In [68]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 63535 entries, 66494 to 121839
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Fecha Inicial   63535 non-null  datetime64[ns]
 1   Monto           63535 non-null  int32         
 2   Edad Solicitud  63535 non-null  int32         
 3   Genero          63535 non-null  object        
 4   Dependientes    63535 non-null  int32         
 5   Ingresos        63535 non-null  int32         
 6   Egresos         63535 non-null  int32         
 7   Ciclo           63535 non-null  int32         
 8   Monto_Ant       63535 non-null  int32         
 9   Frecuencia_Ant  63535 non-null  float64       
 10  Incobrable      63535 non-null  int32         
 11  Score_Ant       63535 non-null  float64       
 12  Score Agt       63535 non-null  float64       
 13  Mes inicial     63535 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int32(8), int64

In [69]:
data.dtypes

Fecha Inicial     datetime64[ns]
Monto                      int32
Edad Solicitud             int32
Genero                    object
Dependientes               int32
Ingresos                   int32
Egresos                    int32
Ciclo                      int32
Monto_Ant                  int32
Frecuencia_Ant           float64
Incobrable                 int32
Score_Ant                float64
Score Agt                float64
Mes inicial                int64
dtype: object

In [70]:
data.head()
#data.to_excel(pkconfig.data_file_name)
data.to_csv("./data/" + pkconfig.data_file_name)

In [71]:
data_tracker.to_dataframe().to_csv("./data/" + pkconfig.conf_column + "_data_tracking.csv")

In [72]:
features_to_drop


['pagos_realizados_actual',
 'pagado_cred_ant',
 'vida_ant',
 'id_fecha_finalizada_ant',
 'tarifa_ant',
 'Categoria',
 'sem_inicio',
 'id_credito_ant',
 'fecha_finalizada',
 'Plazo',
 'Unidad',
 'id_fecha_impresion_actual',
 'CP Aval',
 'vida_actual',
 'id_fecha_impresion_ant',
 'id_cliente',
 'Score CC',
 'CP',
 'score_agente',
 'id_fecha_finalizada_actual',
 'id_agente',
 'semanas',
 'referencia',
 'score']

In [73]:
time_now = datetime.now()

format_date_hour = "%Y-%m-%d %H:%M:%S"
date_and_hour = time_now.strftime(format_date_hour)

# Imprimir la fecha y hora
print("Ended at:", date_and_hour)

Ended at: 2026-11-05 19:48:45


In [74]:
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for RNV_Agt_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for RNV_Agt_CS,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,64759,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
7,Cleaning na floats,63739,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
8,Cleaning numeric data min max,63535,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [75]:
data.head(60)

,Fecha Inicial,Monto,Edad Solicitud,Genero,Dependientes,Ingresos,Egresos,Ciclo,Monto_Ant,Frecuencia_Ant,Incobrable,Score_Ant,Score Agt,Mes inicial
id_solicitud,,,,,,,,,,,,,,
66494,2022-12-30,11000,24,Femenino,0,12400,1000,8,9000,1.063799,0,0.00000,0.000000,12
66857,2023-01-03,14000,45,Femenino,0,7800,5300,8,12000,1.063799,0,0.00000,0.058820,1
72961,2023-02-21,5000,29,Femenino,0,12000,4000,2,3000,1.198630,1,0.00000,0.000000,2
66533,2022-12-30,7000,35,Masculino,0,7000,200,4,5000,1.202552,0,0.00000,0.000000,12
69435,2023-01-23,6000,65,Femenino,0,10000,3500,3,5000,1.482173,0,0.00000,0.000000,1
70265,2023-02-01,7000,46,Femenino,2,10000,9000,3,5000,1.488706,0,0.00000,0.058820,2
72707,2023-02-15,7000,26,Femenino,1,16000,7500,2,5000,1.134702,0,0.00000,0.179400,2
73397,2023-02-21,7000,19,Femenino,1,12500,3000,4,7000,1.001202,0,0.00000,0.167333,2
75199,2023-03-06,9000,20,Femenino,0,13000,4000,4,7000,1.123796,0,0.00000,0.163953,3


In [76]:
data = data.reset_index(drop=True)

In [77]:
data.tail(60)

,Fecha Inicial,Monto,Edad Solicitud,Genero,Dependientes,Ingresos,Egresos,Ciclo,Monto_Ant,Frecuencia_Ant,Incobrable,Score_Ant,Score Agt,Mes inicial
63475,2025-02-05,16000,55,Femenino,0,12000,2900,4,9000,1.847159,1,1.07378,0.000000,2
63476,2025-02-11,13500,24,Masculino,0,7600,7000,7,5500,0.850926,1,0.89532,0.000000,2
63477,2025-02-13,9000,52,Femenino,0,9000,6000,4,5000,1.195557,1,1.01927,0.000000,2
63478,2025-02-26,6000,36,Masculino,1,16000,8000,4,5000,1.134417,1,0.99674,0.000000,2
63479,2025-03-05,11500,64,Masculino,0,0,7000,2,3000,1.413978,1,0.96661,0.000000,3
63480,2025-03-04,10000,40,Femenino,1,12000,5000,6,3000,0.998102,1,0.86371,0.000000,3
63481,2025-03-07,13000,23,Masculino,1,20000,5000,5,5000,1.377879,1,1.03189,0.000000,3
63482,2025-03-12,7500,64,Femenino,0,8000,5000,3,5500,1.134568,1,0.95955,0.010054,3
63483,2025-03-19,6000,35,Masculino,2,0,7200,2,3000,1.060484,1,1.00000,0.009689,3
63484,2025-03-25,17000,24,Femenino,0,12000,7000,5,10000,1.063646,1,1.00000,0.009267,3


In [78]:
# Imprimir la hora actual
print("Finalizado a las:", datetime.now().strftime("%H:%M:%S"))

# Hacer un sonido
if sys.platform.startswith("win"):
    import winsound
    winsound.Beep(1000, 500)  # Frecuencia de 1000 Hz, duración de 500 ms
    winsound.Beep(500, 500) 
    winsound.Beep(1000, 500) 
else:
    print('\a')  # Hace un beep en algunos sistemas UNIX
    
print('Configuration : ', pkconfig.conf_column)
pkdp.print_sizes(data)

Finalizado a las: 19:48:46
Configuration :  RNV_Agt_CS
Rows: 63535
Columns: 14
Max index: 63534
